# News vector RAG — interactive tour

The deep **`AgensgraphVector`** showcase: real news (CC-News) in a pgvector HNSW
store with a fulltext keyword index, queried five ways — ending in a LangChain
**LCEL RAG chain**.

**Prerequisite — ingest first:**

```bash
cd langchain
NEWS_LIMIT=10000 .venv/bin/python examples/demos/03_news_vector_rag/ingest.py
```

In [1]:
import sys, pathlib

HERE = pathlib.Path.cwd()                  # .../03_news_vector_rag
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                 # demos root, for _common
sys.path.insert(0, str(HERE))              # this dir, for `rag`

import pandas as pd
import rag                                  # store construction + helpers live here
from _common import agens, config
from _common.models import get_llm
from langchain_agensgraph.vectorstores.agensgraph_vector import SearchType, HybridSearchConfig

vector = rag._vec(SearchType.VECTOR)                 # vector + filtered search
hybrid = rag._vec(SearchType.HYBRID, keyword="keyword")  # vector + keyword (RRF)
graph = agens.make_graph("news", create=False, refresh_schema=False)

def hits_df(hits):
    return pd.DataFrame([
        {"score": round(s, 3), "title": d.metadata.get("title", "")[:55],
         "domain": d.metadata.get("domain", ""), "date": d.metadata.get("date", "")}
        for d, s in hits
    ])

print("connected to", config.url().split("@")[-1])

connected to localhost:55432/agensgraph_demos


## The corpus

In [2]:
n = graph.query('MATCH (n:"Article") RETURN count(n) AS c')[0]["c"]
rng = graph.query('MATCH (n:"Article") WHERE n.date IS NOT NULL RETURN min(n.date) AS lo, max(n.date) AS hi')[0]
print(f"{n:,} chunks · dates {rng['lo']} .. {rng['hi']}")
pd.DataFrame(graph.query(
    'MATCH (n:"Article") RETURN n.domain AS domain, count(*) AS chunks ORDER BY chunks DESC LIMIT 8'
))

89,445 chunks · dates 2018-05-30 .. 2017-03-10


,domain,chunks
0,nationalpost.com,9660
1,www.taiwannews.com.tw,8275
2,abcnews.go.com,6812
3,www.nigeriatoday.ng,4917
4,www.wave3.com,3698
5,www.yahoo.com,3243
6,www.nytimes.com,3189
7,www.ocregister.com,2197


## (a) Vector semantic search (HNSW)

In [3]:
hits_df(vector.similarity_search_with_score("artificial intelligence and machine learning", k=5))

,score,title,domain,date
0,0.633,Top 5: Things AI might actually be good for,www.techrepublic.com,2018-02-02
1,0.622,Teaching Self-Learning Machines to Forget,www.industryweek.com,2017-12-11
2,0.614,Why some of the world's biggest companies are ...,www.techrepublic.com,2017-12-11
3,0.603,Teaching Self-Learning Machines to Forget,www.industryweek.com,2017-12-11
4,0.575,A Team of MIT Scientists Taught an AI to Get E...,www.yahoo.com,2017-12-11


## (b) Metadata-filtered search

MongoDB-style filter operators on the chunk metadata.

In [4]:
domains = [r["domain"] for r in graph.query(
    'MATCH (n:"Article") RETURN n.domain AS domain, count(*) AS c ORDER BY c DESC LIMIT 3')]
lo = graph.query('MATCH (n:"Article") RETURN min(n.date) AS lo')[0]["lo"]
flt = {"$and": [{"domain": {"$in": domains}}, {"date": {"$gte": lo}}]}
print("filter =", flt)
hits_df(vector.similarity_search_with_score("business and technology", k=5, filter=flt))

filter = {'$and': [{'domain': {'$in': ['nationalpost.com', 'www.taiwannews.com.tw', 'abcnews.go.com']}}, {'date': {'$gte': '2018-05-30'}}]}


,score,title,domain,date
0,0.470,Top Insights on the Cloud Migration Services M...,www.taiwannews.com.tw,2018-05-30
1,0.459,Top Factors Driving the Global Motorcycle Head...,www.taiwannews.com.tw,2018-05-31
2,0.447,Global Consumer NAS Market - Growth Analysis a...,www.taiwannews.com.tw,2018-05-30
3,0.430,Global Food and Beverage Coding and Marking Eq...,www.taiwannews.com.tw,2018-05-31
4,0.428,Growth of the Smartphone Industry to Boost Growth,www.taiwannews.com.tw,2018-07-04


## (c) Hybrid search (vector + keyword, RRF fusion)

`HybridSearchConfig` tunes the reciprocal-rank-fusion weighting.

In [5]:
hits_df(hybrid.similarity_search_with_score(
    "stock market", k=5, hybrid_config=HybridSearchConfig(keyword_weight=2.0)))

,score,title,domain,date
0,0.049,Investment pros staying calm after rate fears ...,www.wafb.com,2018-02-03
1,0.032,Capital market: Investors lose N603.7 bn to ec...,www.nigeriatoday.ng,2017-01-02
2,0.032,Capital market: Investors lose N603.7 bn to ec...,www.nigeriatoday.ng,2017-01-02
3,0.031,Capital market: Investors lose N603.7 bn to ec...,www.nigeriatoday.ng,2017-01-02
4,0.031,Investment pros staying calm after rate fears ...,nationalpost.com,2018-02-03


## (d) effective_search_ratio — over-fetch for recall under a filter

In [6]:
hits_df(vector.similarity_search_with_score(
    "sports", k=5, filter={"date": {"$gte": lo}}, effective_search_ratio=4.0))

,score,title,domain,date
0,0.367,The Latest: Play ended for the day at French O...,www.wave3.com,2018-05-30
1,0.360,ALUNAN RAMBUT CANTIK BERSINAR,herinspirasi.com,2018-05-31
2,0.349,It's Sports and Fitness Day at the White House...,abcnews.go.com,2018-05-30
3,0.347,Win a trip to Singapore to catch Atletico Madr...,www.beinsports.com,2018-06-28
4,0.346,It's Sports and Fitness Day at the White House...,abcnews.go.com,2018-05-30


## (e) RAG — `as_retriever()` in an LCEL chain

The vector store becomes a LangChain retriever, composed with a prompt + LLM
into a cited, grounded answer.

In [7]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

retriever = vector.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('domain','?')} {d.metadata.get('date','')}] "
        f"{d.metadata.get('title','')}\n{d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the news snippets below. Cite the source "
               "domains you rely on. If they don't cover it, say so."),
    ("human", "Question: {question}\n\nNews snippets:\n{context}"),
])
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | get_llm() | StrOutputParser()
)

print(chain.invoke("How is artificial intelligence being used in business?"))

Artificial intelligence (AI) is being utilized in various ways within businesses, including:

1. **Human Resources and Hiring**: AI is increasingly used to manage human employees, particularly in hiring processes. It can analyze resumes, rank candidates, and even assess body language and tone during interviews, as demonstrated by Unilever's use of AI called HireVue. This application helps reduce the time to hire and increases offer acceptance rates (source: www.techrepublic.com).

2. **Decision Making**: AI excels at making data-driven decisions, potentially leading to more objective business outcomes compared to human decision-making, which can be influenced by intuition and biases (source: www.techrepublic.com).

3. **Workforce Reshaping**: Business leaders are encouraged to reimagine work configurations to integrate AI effectively. A significant percentage of executives believe that intelligent technology will be crucial for market differentiation and that roles requiring collaborat

## What you can do with this

One AgensgraphVector store gives you, over the same nodes:

- **semantic search** (HNSW) with scores + metadata,
- **metadata-filtered** retrieval (dates, domains, ranges),
- **hybrid** vector+keyword retrieval (RRF, tunable),
- **recall tuning** via `effective_search_ratio`,
- a drop-in LangChain **retriever** for any LCEL RAG chain or agent.

```python
chain.invoke("your question here")
vector.similarity_search("a topic", k=8, filter={"date": {"$gte": "2018-01-01"}})
```

Close the shared pool when done: `agens.close()`